# Example 2: diagnostic windows, masks, and simple local line fits

In Example 1 we used a clean benchmark spectrum. Here we deliberately switch to a more difficult bundled X-SHOOTER UVB spectrum. We will look at masks, warning regions, and local line diagnostics.

The goal is to learn how Spyctres can help you inspect what parts of a spectrum are useful before launching a full PHOENIX fit.

## What this example teaches

- how Spyctres names known diagnostic lines and windows;
- how warning-only regions differ from pixels actually excluded from fitting;
- how local `fit_line()` diagnostics can succeed numerically while still being physically inadequate for broad features.

## Requirements

Bundled data only. This notebook does not require PHOENIX.

## Expected outputs

A spectrum/mask plot, a diagnostic-window plot, a local line-fit comparison table, and local line diagnostic plots.

Note: A local Gaussian line fit is not an atmospheric-parameter fit and is not a precision model for broad Balmer wings. it is a quick diagnostic.


## 1. Import Spyctres and choose the bundled spectrum

Again we use a reader name, not just an instrument name. `xshooter_merge1d` tells Spyctres how to read this merged one-dimensional X-SHOOTER FITS product and what default wavelength/frame assumptions are appropriate for that product.


In [ ]:
import Spyctres as sp

# Path to the bundled X-SHOOTER UVB example spectrum.
spectrum_path = sp.example_data_path("TOO_Gaia21ccu_SCI_SLIT_FLUX_MERGE1D_UVB.fits")

# Reader profile for merged 1D X-SHOOTER FITS products.
reader = "xshooter_merge1d"

print("Reader used here:")
sp.get_reader_info(reader)


## 2. Read and inspect the spectrum

The first plot is intentionally broad. It helps you spot gaps, bad edges, unusual continuum shape, or obvious artifacts before doing anything more clever.


In [ ]:
# Read the spectrum into Spyctres' common container.
spec = sp.read_spectrum(spectrum_path, reader=reader)

# Print a compact container summary.
print(spec.summary())

# Plot the loaded spectrum before masks or fits.
sp.plot_spectrum(
    spec,
    title="Example 2: loaded X-SHOOTER UVB spectrum",
)


## 3. Ask Spyctres which regions are worth inspecting

The selected windows are advisory. They are useful places to look because they overlap the spectrum and contain common classification or line-shape diagnostics.

The orange overlays still do not change the data. They are visual guideposts.


In [ ]:
# Ask for several candidate diagnostic regions that overlap this spectrum.
windows = sp.select_diagnostic_windows(spec, max_windows=8)

# Print the ranked list so we know what Spyctres selected and why.
print(windows.summary_text(max_rows=8))

# Plot the full spectrum with the candidate windows highlighted.
sp.plot_diagnostic_windows(
    spec,
    selection=windows,
    title="Example 2: suggested diagnostic windows",
)


## 4. Build explicit masks and warnings

`sp.build_mask()` returns a `MaskBundle`. This is a review object: it records what would be excluded, what is only a warning, and how many pixels remain usable.

Important convention: `reviewed_mask.valid_mask` is `True` where a pixel is usable. Warning regions are not applied unless you explicitly ask for that behavior.

Archive/product bad regions come either from metadata attached by the reader, such as `segment.meta["archive_mask_catalog"]`, or from an explicitly requested built-in archive profile. 

You can inspect `reviewed_mask.summary_text()` to see whether any explicit archive masks were actually applied; `exclusion masks: none` means no archive/product exclusion was added.


In [ ]:
# First, build a warning-only bundle. This is useful for review because it shows
# known risky regions without excluding them from later calculations.
warning_bundle = sp.build_mask(
    spec,
    archive="warn",
    tellurics="warn",
    dibs=False,
)
print(warning_bundle.summary_text())

# Now build a reviewed mask bundle.
# If the reader attached archive/product bad-region metadata, archive="mask"
# will turn those regions into explicit exclusions. For this bundled
# X-SHOOTER file no archive bad-region catalogue is attached, so this step
# mainly preserves the spectrum's own valid-mask/finite-data checks.
# Tellurics and known Diffuse Interstellar Bands (DIBs) remain warnings only.
reviewed_mask = sp.build_mask(
    spec,
    archive="mask",
    tellurics="warn",
    dibs=False,
)
print()
print(reviewed_mask.summary_text())

# Plot the same diagnostic windows with the reviewed mask overlay.
sp.plot_diagnostic_windows(
    spec,
    selection=windows,
    mask=reviewed_mask,
    show_nonstellar=True,
    title="Example 2: reviewed mask overlay",
)


### What does this mean for fitting?

The orange diagnostic-window overlays show spectral regions that Spyctres recommends inspecting.

The mask bundle is separate. Pixels are excluded from fitting only if they are
`False` in `reviewed_mask.valid_mask`, or if explicit exclusion masks from the
bundle are passed to a fitting function.

In this example, `archive="mask"` would apply archive/product bad-region masks
if the reader had attached such metadata. In this case, we don't have any such information, so no archive/product exclusion is added.

Telluric regions are set to warning-only, and DIBs are not masked because we have explicitly set `dibs=False`. If you want to mask those as well, set `tellurics='mask'` and `dibs=True` when calling `sp.build_mask()`.

One caveat: tellurics are fixed in the observer/topocentric frame, while DIBs are interstellar and usually not tied to the stellar rest frame. So for precision RV work, it is best to check the spectrum’s wavelength frame before blindly masking these regions. 

You can check what was actually excluded with:

```python
print(reviewed_mask.summary_text())
```

## 5. Fit a few local lines as diagnostics

So far we have used broad diagnostic windows. For local line fitting, Spyctres
also has a small catalogue of named spectral lines. Here we pass names such as
`"Hgamma"` or `"Mg II 4481"` instead of wavelength ranges.

This is different from the diagnostic-window catalogue: a window is a broad
region for inspection, while a named line fit targets one feature and fits a
simple local profile plus continuum.

You can list the known names with `sp.list_known_lines()`, or ask for more
detail with `sp.list_known_lines(details=True)`.


In [ ]:
# List known local lines in the blue optical range used here.
for line in sp.list_known_lines(wmin=3800, wmax=5200, details=True):
    print(line["name"])
    
# Ask Spyctres for the definition of one known line.
sp.known_line_spec("Hgamma")

In [ ]:
# General help for the local-line fitting API.
sp.describe_public_function("fit_line")


In [ ]:
# let's choose a few common optical lines covered by this UVB spectrum.
line_names = ["Hdelta", "Hgamma", "Hbeta", "Mg II 4481"]

Important: `fit_lines()` is a **local diagnostic fitter**, not a full physical
stellar-atmosphere fit. It fits each named feature with a simple local profile
plus a low-order continuum.

That is useful for narrow or moderately isolated features, for checking line
centres, mask behavior, and obvious wavelength problems. It is too simple for broad Balmer lines, whose wings are not Gaussian and may require a proper stellar-atmosphere model such as the PHOENIX fit used later.

In [ ]:
# Run simple local diagnostic fits using the reviewed valid-mask convention.
# This is not a physical Balmer-profile fit; check chi2_red and flags.
# sp.fit_lines() fits a simple local Gaussian line plus a low-order continuum.
line_results = sp.fit_lines(
    spec,
    line_names,
    valid_mask=reviewed_mask.valid_mask,
)

# Compare local RV/EW/chi2 in one compact table.
line_comparison = sp.compare_line_fits(line_results, labels=line_names)
print(line_comparison.summary_text())

# The companion plot shows the same diagnostic quantities visually.
sp.plot_line_fit_comparison(line_comparison)

# Plot one representative line fit.
sp.plot_line_fit(line_results[-1])  # Mg II 4481

In [ ]:
# Take a look at another line.
sp.plot_line_fit(line_results[1])  # Hgamma: useful failure example

`Hgamma` is not a narrow Gaussian line. It has broad Balmer wings, probably pressure/Stark-broadened and affected by continuum placement.

In general, the Balmer lines are broad and are not well described by the simple local Gaussian model used by `fit_line()`. It catches the center-ish part, but completely misses the broad wings.

The continuum estimate is also poor because the “continuum” side of the fitting window is still inside the broad Balmer absorption wing.

`success=True` here only means that the optimizer converged, not that the model is scientifically adequate. The `high_chi2_red` flag is basically saying “do not trust this local Gaussian fit.”

For Balmer-line classification, we need to use diagnostic windows and the PHOENIX full-spectrum/window fit, not a single Gaussian local fit.

## 6. Check whether the local continuum choice matters

If a local fit changes substantially when the continuum order changes, that line is telling you that the local normalization is important. That is a diagnostic result, not a failure.

`LineFitConfig(continuum_order=...)` controls the local continuum model used by the simple line fitter. Currently Spyctres allows low-order Legendre continua:

- `continuum_order=0`: constant continuum
- `continuum_order=1`: linear continuum
- `continuum_order=2`: quadratic continuum
- `continuum_order=3`: cubic continuum

Higher order is not always better. A more flexible continuum can reduce χ² by
absorbing broad line wings or calibration structure, but that may hide the fact that the local line-profile model is too simple. For this reason, local
continuum changes should be treated as a sensitivity check.


In [ ]:
# Fit Hgamma twice with two simple continuum choices.
hgamma_linear = sp.fit_line(
    spec,
    "Hgamma",
    config=sp.LineFitConfig(continuum_order=1),
    valid_mask=reviewed_mask.valid_mask,
)

hgamma_quadratic = sp.fit_line(
    spec,
    "Hgamma",
    config=sp.LineFitConfig(continuum_order=2),
    valid_mask=reviewed_mask.valid_mask,
)

# Compare the two local continuum assumptions.
continuum_check = sp.compare_line_fits(
    [hgamma_linear, hgamma_quadratic],
    labels=["linear continuum", "quadratic continuum"],
)

print(continuum_check.summary_text())

# Same-line variants can also be compared with the compact metric plot.
sp.plot_line_fit_comparison(continuum_check)

In [ ]:
# Plot them
sp.plot_line_fit(hgamma_linear)
sp.plot_line_fit(hgamma_quadratic)

Both fits converged, but both are still flagged with `high_chi2_red`. The
quadratic continuum reduces χ², which tells us that continuum placement matters for this broad Balmer feature, but it still does not make the simple Gaussian line model physically adequate.

This is the key lesson: changing the local continuum can improve the numerical
fit while the underlying line-profile model is still wrong. Only use these simple local fits as diagnostics of continuum/model sensitivity, not as final measurements of the line wings or stellar parameters.

## What to try next

Examples 3 and 4 keep the same spectrum and show how to improve a PHOENIX setup one assumption at a time: windows, masks, resolution, continuum degree, and search budget.

For help on local line fitting:


In [ ]:
sp.describe_public_function("fit_line")